In [ ]:
#! mlflow server -p 5001

In [ ]:
import mlflow
from openai import OpenAI
from loguru import logger
from mlflow.entities import SpanType
import time

In [ ]:
PORT = "5001"
EXPERIMENT_NAME = "Chatbot"
TRACE_ID = "id-12345"
TRACE_KEY = "chatbot_version"
TRACE_VALUE = "1.0"

OLLAMA_API_URL = "http://localhost:11434/v1"
LLM_MODEL = "gpt-oss:20b"
TEMPERATURE = 0.1
MAX_TOKENS = 100

In [ ]:
mlflow.openai.autolog()
mlflow.set_tracking_uri(f"http://localhost:{PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
class Agent:
    def __init__(self, system=""):
        self.client = self._set_llm(OLLAMA_API_URL)
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})
            logger.info(f"System message set: {system}")

    def _set_llm(self, url):
        try:
            logger.info(f"Ollama API URL: {url}")
            return OpenAI(base_url=url, 
                          api_key="dummy")
        except Exception as e:
            print(f"Error initializing OpenAI client: {e}")
            return None

    @mlflow.trace(span_type=SpanType.CHAIN)
    def __call__(self, user_input):
        try:
            self.messages.append({"role": "user", "content": user_input})
            logger.info(f"User message added: {user_input}")
            result = self.execute()
            self.messages.append({"role": "assistant", "content": result})
            logger.info(f"Assistant response added: {result}")
            return result
        except Exception as e:
            logger.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def execute(self):
        try:
            response = self.client.chat.completions.create(
                        model=LLM_MODEL,
                        messages=self.messages,
                        temperature=TEMPERATURE,
                        max_tokens=MAX_TOKENS,
                    )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Error during agent execution: {e}")
            return "An error occurred while processing your request."

## AI Chatbot

In [ ]:
def start_session():
    agent = Agent(system="You are a helpful assistant.")
    while True:
        user_query = input("You: ")
        if user_query == "BYE":
            break
        response = agent(user_query)

with mlflow.start_run(run_name="Chatbot Session"):
    start_session()